In [15]:
import tsl
import torch
import pandas as pd

In [16]:
df = pd.read_csv('../data/EWZ_cleaned.csv', index_col=0)

df.head()

,Wärmezähler UST 10 Spitalstrasse 6 Regionalspital Surselva WZ11,Wärmezähler Ölkessel Spital WZ10,Wärmezähler Holzkessel WT01 Direkt WZ1,Wärmezähler UST 11 Via Schlifras 46,Wärmezähler UST 12 Via Schlifras 48,Wärmezähler UST 13 Via Schlifras 50,Wärmezähler UST 14 Via Schlifras 54,Wärmezähler UST 15 Solaranlage BWW,Wärmezähler UST 18 Schulstrasse 13,"Wärmezähler UST 19 Städtlistrasse 16 u. St.Margrethenplatz, Evang. Kirchgemeinde",...,Wärmezähler UST 80 Via Santeri 6,Wärmezähler UST 81 Via Santeri 4,"Wärmezähler UST 82 Via S. Clau Sut 2,4",Wärmezähler UST 83 Glennerstrasse 18,Wärmezähler Spitalstrasse 7 UST 84,Wärmezähler Spitalstrasse 8 UST 85,Wärmezähler UST 86 Valserstrasse 7,Wärmezähler UST 87 Via Hans Erni 6,Wärmezähler Via Schlifras 62/64 UST 89,Wärmezähler Via Sogn Martin 1+3 UST 91
Time,,,,,,,,,,,,,,,,,,,,,
2023-11-01 00:00:00,0.0252,0.0,0.0,0.000,0.0021,0.000,0.0,0.0,0.0,0.0,...,0.0,0.000,0.000,0.000,0.019,0.004,0.004,0.000,0.004,0.003
2023-11-01 00:15:00,0.0278,0.0,0.0,0.000,0.0000,0.000,0.0,0.0,0.0,0.0,...,0.0,0.001,0.004,0.006,0.000,0.000,0.000,0.002,0.000,0.000
2023-11-01 00:30:00,0.0236,0.0,0.0,0.000,0.0000,0.031,0.0,0.0,0.0,0.0,...,0.0,0.000,0.000,0.000,0.007,0.002,0.004,0.000,0.008,0.004
2023-11-01 00:45:00,0.0211,0.0,0.0,0.031,0.0000,0.000,0.0,0.0,0.0,0.0,...,0.0,0.001,0.004,0.007,0.000,0.000,0.000,0.001,0.000,0.000
2023-11-01 01:00:00,0.0161,0.0,0.0,0.000,0.0000,0.000,0.0,0.0,0.0,0.0,...,0.0,0.000,0.000,0.000,0.014,0.004,0.005,0.000,0.008,0.004


In [17]:
from tsl.data import SpatioTemporalDataset

torch_dataset = SpatioTemporalDataset(target=df,
                                      horizon=12,
                                      window=12,
                                      stride=1)

In [18]:
from tsl.data.datamodule import (SpatioTemporalDataModule,
                                 TemporalSplitter)
from tsl.data.preprocessing import StandardScaler # Dont need it to add standard scaler in csv preprocessing

# Normalize data using mean and std computed over time and node dimensions
scalers = {'target': StandardScaler(axis=(0, 1))}

# Split data sequentially:
#   |------------ dataset -----------|
#   |--- train ---|- val -|-- test --|
splitter = TemporalSplitter(val_len=0.1, test_len=0.3)

dm = SpatioTemporalDataModule(
    dataset=torch_dataset,
    scalers=scalers,
    splitter=splitter,
    batch_size=64,
)

In [19]:


from lib.nn.encoders.corel_encoder import CoRelEncoder
from lib.nn.decoder.base_decoder import BaseDecoder
from lib.nn.encoder_decoder_model import EncoderDecoderModel


In [20]:
def print_model_size(model):
    tot = sum([p.numel() for p in model.parameters() if p.requires_grad])
    out = f"Number of model ({model.__class__.__name__}) parameters:{tot:10d}"
    print("=" * len(out))
    print(out)

hidden_size = 32   #@param
temporal_layers = 1     #@param
spatial_layers = 1     #@param
conv_type = "diffconv"  #@param ["diffconv", "graphconv"]
temporal_type = "gru"   #@param ["gru", "lstm"]

input_size = torch_dataset.n_channels   # 1 channel
n_nodes = torch_dataset.n_nodes         # 207 nodes
horizon = torch_dataset.horizon         # 12 time steps


stgnn = EncoderDecoderModel(
    input_size=input_size,
    output_size=input_size,
    horizon=horizon,
    encoder_class=CoRelEncoder,
    encoder_kwargs={'gnn_layers': spatial_layers, 'temporal_layers': temporal_layers,
                    'hidden_size': hidden_size, 'n_instances': n_nodes, 'emb_size': hidden_size,
                    'n_neighbors':2, 'conv_type': conv_type,},
    decoder_class=BaseDecoder,
    decoder_kwargs={},
    exog_size= 0,
)




print_model_size(stgnn)

Number of model (EncoderDecoderModel) parameters:     17120


In [21]:
from tsl.metrics.torch import MaskedMAE, MaskedMAPE
from tsl.engines import Predictor

loss_fn = MaskedMAE()

metrics = {'mae': MaskedMAE(),
           'mape': MaskedMAPE(),
           'mae_at_15': MaskedMAE(at=2),  # '2' indicates the third time step,
                                          # which correspond to 15 minutes ahead
           'mae_at_30': MaskedMAE(at=5),
           'mae_at_60': MaskedMAE(at=11)}

# setup predictor
predictor = Predictor(
    model=stgnn,                   # our initialized model
    optim_class=torch.optim.Adam,  # specify optimizer to be used...
    optim_kwargs={'lr': 0.001},    # ...and parameters for its initialization
    loss_fn=loss_fn,               # which loss function to be used
    metrics=metrics                # metrics to be logged during train/val/test
)

In [22]:
from pytorch_lightning.loggers import TensorBoardLogger

logger = TensorBoardLogger(save_dir="logs", name="tsl_intro", version=0)

In [23]:
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint

checkpoint_callback = ModelCheckpoint(
    dirpath='logs',
    save_top_k=1,
    monitor='val_mae',
    mode='min',
)

trainer = pl.Trainer(max_epochs=1,
                     logger=logger,
                     limit_train_batches=10,  # end an epoch after 10 updates
                     callbacks=[checkpoint_callback])

trainer.fit(predictor, datamodule=dm)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
c:\Users\matya\OneDrive\Bureau\Semester_project\SemesterProject\venv\lib\site-packages\pytorch_lightning\callbacks\model_checkpoint.py:751: Checkpoint directory C:\Users\matya\OneDrive\Bureau\Semester_project\SemesterProject\src\logs exists and is not empty.

  | Name          | Type                | Params | Mode 
--------------------------------------------------------------
0 | loss_fn       | MaskedMAE           | 0      | train
1 | train_metrics | MetricCollection    | 0      | train
2 | val_metrics   | MetricCollection    | 0      | train
3 | test_metrics  | MetricCollection    | 0      | train
4 | model         | EncoderDecoderModel | 17.1 K | train
--------------------------------------------------------------
17.1 K    Trainable params
0         Non-trainable params
17.1 K    Total params
0.068     Total estimated model params size (MB)
35        Modules in train mode

Sanity Checking DataLoader 0:   0%|          | 0/2 [00:00<?, ?it/s]

c:\Users\matya\OneDrive\Bureau\Semester_project\SemesterProject\venv\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
c:\Users\matya\OneDrive\Bureau\Semester_project\SemesterProject\src\lib\nn\utils.py:12: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\python_variable_indexing.cpp:322.)
  emb = emb[[None] * (x.ndim - emb.ndim)]


c:\Users\matya\OneDrive\Bureau\Semester_project\SemesterProject\venv\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
c:\Users\matya\OneDrive\Bureau\Semester_project\SemesterProject\venv\lib\site-packages\pytorch_lightning\loops\fit_loop.py:310: The number of training batches (10) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Epoch 0: 100%|██████████| 10/10 [00:05<00:00,  1.76it/s, v_num=0, val_mae=0.00947, val_mae_at_15=0.009, val_mae_at_30=0.00963, val_mae_at_60=0.00934, val_mape=3.11e+4, train_mae=0.00782, train_mae_at_15=0.00607, train_mae_at_30=0.00591, train_mae_at_60=0.00793, train_mape=6.42e+4]

`Trainer.fit` stopped: `max_epochs=1` reached.


Epoch 0: 100%|██████████| 10/10 [00:05<00:00,  1.74it/s, v_num=0, val_mae=0.00947, val_mae_at_15=0.009, val_mae_at_30=0.00963, val_mae_at_60=0.00934, val_mape=3.11e+4, train_mae=0.00782, train_mae_at_15=0.00607, train_mae_at_30=0.00591, train_mae_at_60=0.00793, train_mape=6.42e+4]


In [24]:
# predictor.load_model(checkpoint_callback.best_model_path)
# predictor.freeze()

trainer.test(predictor, datamodule=dm)

c:\Users\matya\OneDrive\Bureau\Semester_project\SemesterProject\venv\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Testing DataLoader 0: 100%|██████████| 309/309 [00:14<00:00, 21.97it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_loss           0.00493971910327673
        test_mae           0.004939720034599304
     test_mae_at_15        0.0036781637463718653
     test_mae_at_30        0.004387491848319769
     test_mae_at_60        0.004979150835424662
        test_mape             40529.24609375
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'test_mae': 0.004939720034599304,
  'test_mae_at_15': 0.0036781637463718653,
  'test_mae_at_30': 0.004387491848319769,
  'test_mae_at_60': 0.004979150835424662,
  'test_mape': 40529.24609375,
  'test_loss': 0.00493971910327673}]